In [1]:
import sys
print(sys.executable)

c:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\Training Instant\Week 2\CheatingTrackingSystem\.venv\Scripts\python.exe


In [2]:
from ultralytics import YOLO

print("Ultralytics imported successfully!")

Ultralytics imported successfully!


In [4]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    save=True,
    conf=0.25
)

print("Detection completed!")
print("Number of detected objects:", len(results[0].boxes))
print("Saved results to:", results[0].save_dir)


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 c:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\bus.jpg: 640x480 4 persons, 1 bus, 156.4ms
Speed: 12.1ms preprocess, 156.4ms inference, 26.5ms postprocess per image at shape (1, 3, 640, 480)
Results saved to C:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\runs\detect\predict-2
Detection completed!
Number of detected objects: 5
Saved results to: C:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\runs\detect\predict-2


In [26]:
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\Training Instant\Week 2\CheatingTrackingSystem"
)

VIDEO_DIR = PROJECT_DIR / "test_videos"

print("Project folder exists:", PROJECT_DIR.exists())
print("Videos folder exists:", VIDEO_DIR.exists())

if VIDEO_DIR.exists():
    print("\nVideo files:")
    for file in sorted(VIDEO_DIR.iterdir()):
        if file.is_file():
            print("-", file.name)
else:
    print("\nمش لاقي فولدر test_videos في المسار ده.")

Project folder exists: True
Videos folder exists: True

Video files:
- ClassRoomExam.mp4
- Looking_at_mobile.mp4.mp4
- looking_at_paper.mp4.mp4


In [27]:
import cv2
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\Training Instant\Week 2\CheatingTrackingSystem"
)

video_path = PROJECT_DIR / "test_videos" / "ClassRoomExam.mp4"

cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    print("ERROR: Could not open the new video.")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print("Video opened successfully!")
    print(f"Name: {video_path.name}")
    print(f"Resolution: {width} x {height}")
    print(f"FPS: {fps:.2f}")
    print(f"Frames: {total_frames}")
    print(f"Duration: {total_frames / fps:.2f} seconds" if fps > 0 else "Duration: unknown")

cap.release()

Video opened successfully!
Name: ClassRoomExam.mp4
Resolution: 768 x 432
FPS: 24.00
Frames: 488
Duration: 20.33 seconds


In [13]:

%pip install ipywidgets


   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 4.2 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.2 MB 2.6 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 2.4 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 2.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:

from pathlib import Path
from datetime import datetime
import csv
import time

import cv2
import numpy as np
import mediapipe as mp
import ipywidgets as widgets

from IPython.display import display, clear_output
from ultralytics import YOLO


# 1. Project folders
PROJECT_DIR = Path(
    r"C:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\Training Instant\Week 2\CheatingTrackingSystem"
)

VIDEO_DIR = PROJECT_DIR / "test_videos"
OUTPUT_DIR = PROJECT_DIR / "output"
MODEL_PATH = PROJECT_DIR / "models" / "face_landmarker.task"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Find videos
videos = sorted(
    p for p in VIDEO_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}
)

if not videos:
    raise FileNotFoundError(f"No videos found in {VIDEO_DIR}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Face model not found: {MODEL_PATH}")

video_dropdown = widgets.Dropdown(
    options=[(v.name, str(v)) for v in videos],
    description="Video:",
    layout=widgets.Layout(width="600px")
)

analyze_button = widgets.Button(
    description="Analyze Video",
    button_style="success"
)

output = widgets.Output()


def analyze_video(_):
    with output:
        clear_output(wait=True)

        video_path = Path(video_dropdown.value)
        video_output = OUTPUT_DIR / video_path.stem
        screenshots_dir = video_output / "screenshots"

        screenshots_dir.mkdir(parents=True, exist_ok=True)

        output_video = video_output / "annotated_video.mp4"
        alerts_file = video_output / "alerts.csv"

        print("Selected video:", video_path.name)
        print("Loading YOLO and face models...")

        model = YOLO("yolo11n.pt")

        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            print("ERROR: Could not open video.")
            return

        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        if fps <= 0:
            fps = 25.0

        writer = cv2.VideoWriter(
            str(output_video),
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (width, height)
        )

        if not writer.isOpened():
            cap.release()
            print("ERROR: Could not create output video.")
            return

        # Face detector using the installed MediaPipe Tasks API
        face_options = mp.tasks.vision.FaceLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(
                model_asset_path=str(MODEL_PATH)
            ),
            running_mode=mp.tasks.vision.RunningMode.VIDEO,
            num_faces=10
        )

        frame_count = 0
        phone_detections = 0
        tracked_ids = set()
        alert_rows = []
        saved_alerts = set()

        # Count consecutive frames with a sideways head-turn candidate
        head_turn_counts = {}
        last_timestamp_ms = -1

        print("Analyzing every frame. Please wait...")

        try:
            with mp.tasks.vision.FaceLandmarker.create_from_options(
                face_options
            ) as face_landmarker:

                while True:
                    success, frame = cap.read()

                    if not success:
                        break

                    frame_count += 1

                    # A. Detect and track people and phones
                    result = model.track(
                        source=frame,
                        persist=True,
                        tracker="bytetrack.yaml",
                        classes=[0, 67],
                        conf=0.25,
                        verbose=False
                    )[0]

                    people = []
                    phones = []

                    if result.boxes is not None:
                        boxes = result.boxes
                        xyxy = boxes.xyxy.cpu().numpy()
                        classes = boxes.cls.cpu().numpy().astype(int)

                        if boxes.id is not None:
                            ids = boxes.id.cpu().numpy().astype(int)
                        else:
                            ids = [None] * len(classes)

                        for box, cls, track_id in zip(xyxy, classes, ids):
                            x1, y1, x2, y2 = map(int, box)

                            if cls == 0:
                                people.append(
                                    (x1, y1, x2, y2, track_id)
                                )
                                if track_id is not None:
                                    tracked_ids.add(track_id)

                            elif cls == 67:
                                phones.append((x1, y1, x2, y2))
                                phone_detections += 1

                    # B. Detect facial landmarks on this frame
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(
                        image_format=mp.ImageFormat.SRGB,
                        data=rgb
                    )

                    timestamp_ms = int(frame_count * 1000 / fps)
                    timestamp_ms = max(
                        timestamp_ms, last_timestamp_ms + 1
                    )
                    last_timestamp_ms = timestamp_ms

                    face_result = face_landmarker.detect_for_video(
                        mp_image, timestamp_ms
                    )

                    face_candidates = []

                    if face_result.face_landmarks:
                        for face in face_result.face_landmarks:
                            nose_x = face[1].x * width
                            left_eye_x = face[33].x * width
                            right_eye_x = face[263].x * width

                            eye_span = abs(right_eye_x - left_eye_x)

                            if eye_span > 1:
                                # A rough sideways-head-turn indicator.
                                # This is NOT reliable gaze or cheating detection.
                                eye_midpoint = (
                                    left_eye_x + right_eye_x
                                ) / 2

                                turn_ratio = (
                                    nose_x - eye_midpoint
                                ) / eye_span

                                face_center_x = nose_x
                                face_center_y = face[1].y * height

                                face_candidates.append(
                                    (
                                        face_center_x,
                                        face_center_y,
                                        turn_ratio
                                    )
                                )

                    # C. Draw detected phones
                    for px1, py1, px2, py2 in phones:
                        cv2.rectangle(
                            frame, (px1, py1), (px2, py2),
                            (0, 255, 255), 2
                        )
                        cv2.putText(
                            frame, "Phone detected",
                            (px1, max(py1 - 8, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, (0, 255, 255), 2
                        )

                    # D. Evaluate each tracked person
                    for x1, y1, x2, y2, track_id in people:
                        person_width = max(1, x2 - x1)
                        person_height = max(1, y2 - y1)

                        margin_x = int(person_width * 0.20)
                        margin_y = int(person_height * 0.10)

                        ex1 = x1 - margin_x
                        ey1 = y1 - margin_y
                        ex2 = x2 + margin_x
                        ey2 = y2 + margin_y

                        phone_near_person = any(
                            ex1 <= (px1 + px2) / 2 <= ex2
                            and ey1 <= (py1 + py2) / 2 <= ey2
                            for px1, py1, px2, py2 in phones
                        )

                        # Associate a face with this person box
                        matched_turn_ratio = None

                        for fx, fy, turn_ratio in face_candidates:
                            if x1 <= fx <= x2 and y1 <= fy <= y2:
                                matched_turn_ratio = turn_ratio
                                break

                        head_turn_candidate = False

                        if track_id is not None:
                            if matched_turn_ratio is not None:
                                # Threshold is only an initial experiment.
                                # Camera angle and student position affect it.
                                if abs(matched_turn_ratio) > 0.25:
                                    head_turn_counts[track_id] = (
                                        head_turn_counts.get(track_id, 0) + 1
                                    )
                                else:
                                    head_turn_counts[track_id] = 0

                                # Require several consecutive frames
                                head_turn_candidate = (
                                    head_turn_counts[track_id] >= 8
                                )
                            else:
                                # Avoid accumulating evidence when no face
                                # can be detected in this frame.
                                head_turn_counts[track_id] = 0

                        if phone_near_person:
                            box_color = (0, 0, 255)
                            label = "REVIEW - Phone nearby"
                            behavior = "Phone nearby - review"

                        elif head_turn_candidate:
                            box_color = (0, 0, 255)
                            label = "REVIEW - Head turned"
                            behavior = "Repeated head-turn candidate"

                        else:
                            box_color = (0, 200, 0)
                            label = "Person"
                            behavior = None

                        # Save one screenshot per student and alert type
                        if (
                            behavior is not None
                            and track_id is not None
                            and (track_id, behavior) not in saved_alerts
                        ):
                            now = datetime.now()
                            timestamp = now.strftime("%Y%m%d_%H%M%S_%f")

                            screenshot_path = (
                                screenshots_dir
                                / f"person_{track_id}_{timestamp}.jpg"
                            )

                            # Include the current person's colored box
                            cv2.rectangle(
                                frame, (x1, y1), (x2, y2),
                                box_color, 3
                            )

                            cv2.imwrite(
                                str(screenshot_path), frame
                            )

                            alert_rows.append({
                                "datetime": now.isoformat(
                                    timespec="seconds"
                                ),
                                "student_id": track_id,
                                "behavior": behavior,
                                "confidence": "",
                                "screenshot_path": str(screenshot_path)
                            })

                            saved_alerts.add((track_id, behavior))

                        cv2.rectangle(
                            frame, (x1, y1), (x2, y2),
                            box_color, 2
                        )

                        if track_id is not None:
                            label += f" | ID {track_id}"

                        cv2.putText(
                            frame, label,
                            (x1, max(y1 - 10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.55, box_color, 2
                        )

                    # E. Write this annotated frame to output video
                    writer.write(frame)

                    if frame_count % 100 == 0:
                        print(f"Processed {frame_count} frames...")

        finally:
            cap.release()
            writer.release()

        # F. Save alert log
        fieldnames = [
            "datetime",
            "student_id",
            "behavior",
            "confidence",
            "screenshot_path"
        ]

        with open(
            alerts_file, "w", newline="", encoding="utf-8"
        ) as file:
            csv_writer = csv.DictWriter(
                file, fieldnames=fieldnames
            )
            csv_writer.writeheader()
            csv_writer.writerows(alert_rows)

        print("\nAnalysis completed!")
        print("Frames analyzed:", frame_count)
        print("Tracked people:", len(tracked_ids))
        print("Phone detections:", phone_detections)
        print("Review alerts:", len(alert_rows))
        print("Annotated video:", output_video)
        print("Screenshots folder:", screenshots_dir)
        print("Alert log:", alerts_file)
        print(
            "\nNote: Head-turn alerts are preliminary review signals, "
            "not proof of cheating."
        )


analyze_button.on_click(analyze_video)

display(video_dropdown, analyze_button, output)


Dropdown(description='Video:', layout=Layout(width='600px'), options=(('classroom_exam.mp4.mp4', 'C:\\Users\\A…

Button(button_style='success', description='Analyze Video', style=ButtonStyle())

Output()

In [1]:
%pip install mediapipe

  Using cached mediapipe-1.2.0-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached absl_py-2.5.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached sounddevice-0.5.6-py3-none-win_amd64.whl.metadata (1.4 kB)
  Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached cffi-2.1.1-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
Using cached mediapipe-1.2.0-py3-none-win_amd64.whl (24.5 MB)
Using cached absl_py-2.5.1-py3-none-any.whl (137 kB)
Using cached sounddevice-0.5.6-py3-none-win_amd64.whl (1.0 MB)
Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-win_amd64.whl (53.8 MB)
Using cached cffi-2.1.1-cp312-cp312-win_amd64.whl (185 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:

import mediapipe as mp

print("MediaPipe version:", mp.__version__)
print("Has solutions:", hasattr(mp, "solutions"))
print("Has tasks:", hasattr(mp, "tasks"))
print("MediaPipe location:", mp.__file__)


MediaPipe version: 1.2.0
Has solutions: False
Has tasks: True
MediaPipe location: c:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\Training Instant\Week 2\CheatingTrackingSystem\.venv\Lib\site-packages\mediapipe\__init__.py


In [7]:

from pathlib import Path
import urllib.request

PROJECT_DIR = Path(
    r"C:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\Training Instant\Week 2\CheatingTrackingSystem"
)

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "face_landmarker.task"

if not MODEL_PATH.exists():
    url = (
        "https://storage.googleapis.com/mediapipe-models/"
        "face_landmarker/face_landmarker/float16/1/"
        "face_landmarker.task"
    )

    print("Downloading face landmark model...")
    urllib.request.urlretrieve(url, MODEL_PATH)

print("Model ready:", MODEL_PATH)
print("File size (MB):", round(MODEL_PATH.stat().st_size / (1024 * 1024), 2))


Model ready: C:\Users\ASUS\OneDrive - Ahram Canadian University\Desktop\Training Instant\Week 2\CheatingTrackingSystem\models\face_landmarker.task
File size (MB): 3.58
